In [8]:
!pip install yfinance
import yfinance as yf
from pathlib import Path

data_dir = Path('../data/raw')

# Download SPY daily data up to today
spy = yf.download("SPY", start="1990-01-01", auto_adjust=False, progress=False)

# Save to CSV
output_path = data_dir / "SPY_daily_yahoo_raw.csv"
spy.to_csv(output_path)

print(f"Data saved to: {output_path}")
print(f"Latest date in data: {spy.index[-1].date()}")
print(f"Rows: {len(spy)}")

Data saved to: ..\data\raw\SPY_daily_yahoo_raw.csv
Latest date in data: 2026-05-15
Rows: 8381


In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

# Download latest data
print("Downloading latest SPY data...")
df = yf.download("SPY", start="1990-01-01", auto_adjust=False, progress=False)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

P = df['Adj Close'].values
dates = df.index

# Training data up to end 2024
P_train = df[df.index <= '2024-12-31']['Adj Close'].values

# ============================================================
# Check both strategies
# ============================================================
strategies = [
    {"tau": 200, "q": 0.9, "label": "Strategy A (tau=200, q=0.9)"},
    {"tau": 100, "q": 0.9, "label": "Strategy B (tau=100, q=0.9)"},
]

for s in strategies:
    tau = s["tau"]
    q = s["q"]
    
    # Threshold from training data only
    inc_train = P_train[tau:] - P_train[:-tau]
    thresh = np.quantile(inc_train, q).item()
    
    # Current signal
    inc_now = (P[-1] - P[-1 - tau]).item()
    signal = inc_now >= thresh
    
    print(f"\n{s['label']}")
    print(f"  Current {tau}-day increment: {inc_now:.2f}")
    print(f"  Threshold (90th percentile): {thresh:.2f}")
    print(f"  Signal: {'🟢 BUY SPY' if signal else '🔴 NO TRADE'}")


Strategy A (tau=200, q=0.9)
  Current 200-day increment: 110.05
  Threshold (90th percentile): 37.42
  Signal: 🟢 BUY SPY

Strategy B (tau=100, q=0.9)
  Current 100-day increment: 60.43
  Threshold (90th percentile): 25.58
  Signal: 🟢 BUY SPY


In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import date

# ============================================================
# TRADE LOG
# ============================================================
trades = [
    {
        "instrument": "SPY ARCA",
        "entry_date": "2026-05-15",
        "entry_price": 743.23,
        "shares": 1,
    },
]

# ============================================================
# STRATEGIES TO TRACK
# ============================================================
strategies = [
    {"label": "Strategy A", "tau": 200, "q": 0.9, "r": 0.8},
    {"label": "Strategy B", "tau": 100, "q": 0.9, "r": 0.8},
]

# ============================================================
# Download latest SPY data
# ============================================================
print("Downloading latest SPY data...")
df = yf.download("SPY", start="1990-01-01", auto_adjust=False, progress=False)
df.index = pd.to_datetime(df.index)
df = df.sort_index()

P_train = df[df.index <= '2024-12-31']['Adj Close'].values
spy_2026 = df[df.index >= '2026-01-01']
trading_days = spy_2026.index
current_price = spy_2026['Adj Close'].iloc[-1].item()
today = trading_days[-1]
actual_today = pd.Timestamp(date.today())

print(f"\nActual today:      {actual_today.date()}")
print(f"Last trading day:  {today.date()}")
print(f"Current SPY price: ${current_price:.2f}")

# ============================================================
# Compute r thresholds for each strategy
# ============================================================
r_thresholds = {}
for s in strategies:
    tau = s["tau"]
    r = s["r"]
    P_all = df['Adj Close'].values
    inc_train = P_all[tau:] - P_all[:-tau]
    inc_train = inc_train[:len(P_train) - tau]
    r_thresholds[s["label"]] = np.quantile(inc_train, r).item()

# ============================================================
# Monitor each trade
# ============================================================
for i, trade in enumerate(trades):
    entry_date = pd.Timestamp(trade["entry_date"])
    entry_price = trade["entry_price"]
    shares = trade["shares"]
    instrument = trade["instrument"]

    calendar_days_elapsed = (actual_today - entry_date).days
    pnl = (current_price - entry_price) * shares
    pnl_pct = (pnl / (entry_price * shares)) * 100

    print(f"\n{'=' * 55}")
    print(f"Trade {i+1}: {instrument}")
    print(f"  Entry date:         {entry_date.date()}")
    print(f"  Entry price USD:    ${entry_price:.2f}")
    print(f"  Current price USD:  ${current_price:.2f}")
    print(f"  Shares:             {shares}")
    print(f"  Calendar days held: {calendar_days_elapsed}")
    print(f"  Unrealised P&L:     ${pnl:.2f} ({pnl_pct:.2f}%)")

    # --------------------------------------------------------
    # Strategy checkpoints
    # --------------------------------------------------------
    for s in strategies:
        tau = s["tau"]
        label = s["label"]
        r = s["r"]

        # Trading days elapsed
        days_elapsed = len(trading_days[
            (trading_days > entry_date) & (trading_days <= today)
        ])

        # Remaining trading days
        days_remaining = tau - days_elapsed

        # Exit date estimate
        future_days = trading_days[trading_days > today]
        if days_remaining > 0 and len(future_days) >= days_remaining:
            estimated_exit = future_days[days_remaining - 1]
            exit_str = str(estimated_exit.date())
        elif days_remaining <= 0:
            exit_str = "CHECKPOINT REACHED"
        else:
            exit_str = "Need more data"

        # Target price (r threshold above entry)
        r_thresh = r_thresholds[label]
        target_price = entry_price + r_thresh
        target_met = (current_price - entry_price) >= r_thresh

        print(f"\n  📊 {label} (tau={tau}, q=0.9, r={r})")
        print(f"     Trading days held:  {days_elapsed} / {tau}")
        print(f"     Days remaining:     {days_remaining}")
        print(f"     Estimated exit:     {exit_str}")
        print(f"     Target price:       ${target_price:.2f}")
        print(f"     Target met:         {'✅ YES' if target_met else '❌ Not yet'}")
        print(f"     Status:             {'⚠️  EXIT NOW' if days_remaining <= 0 else '✅ Hold'}")

    print("-" * 55)


Actual today:      2026-05-16
Last trading day:  2026-05-15
Current SPY price: $739.17

Trade 1: SPY ARCA
  Entry date:         2026-05-15
  Entry price USD:    $743.23
  Current price USD:  $739.17
  Shares:             1
  Calendar days held: 1
  Unrealised P&L:     $-4.06 (-0.55%)

  📊 Strategy A (tau=200, q=0.9, r=0.8)
     Trading days held:  0 / 200
     Days remaining:     200
     Estimated exit:     Need more data
     Target price:       $766.32
     Target met:         ❌ Not yet
     Status:             ✅ Hold

  📊 Strategy B (tau=100, q=0.9, r=0.8)
     Trading days held:  0 / 100
     Days remaining:     100
     Estimated exit:     Need more data
     Target price:       $756.79
     Target met:         ❌ Not yet
     Status:             ✅ Hold
-------------------------------------------------------
